# V6.7 — Finding Seed 36 YOLO evaluation on Finding Nemo audit V5

Test all eight Seed 36 YOLO checkpoints (raw/curated × YOLO11n/26n/11m/26m) on the frozen 1,931-image audit V5 test set. Each run is saved locally and uploaded to Camber; the combined summary is stored at the result root.

No inference starts until the **Run selected evaluations** cell is executed.

In [ ]:
from pathlib import Path
import csv, hashlib, json, os, shutil, subprocess, time, zipfile
import yaml

def find_repo(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / 'benchmakring_transmission_line_defect_model').is_dir():
            return candidate
    raise FileNotFoundError('Repository root not found')

REPO = find_repo(Path.cwd())
BENCHMARK = REPO / 'benchmakring_transmission_line_defect_model'
TEST_ROOT = REPO / 'data/working/finding_nemo_test_audit_v5_bundle/data/test_main_v5_full_consensus_assisted'
TEST_YAML = TEST_ROOT / 'dataset.yaml'
WEIGHTS_ROOT = BENCHMARK / 'weights/finding_seed/seed36'
RESULTS_ROOT = BENCHMARK / 'results_seed/finding_seed36_test_audit_v5'
REMOTE_SEED_ROOT = 'stash://bunpmc/projects/finding_seed/seed36'
REMOTE_RESULTS_ROOT = REMOTE_SEED_ROOT + '/test_audit_v5_results'
REMOTE_BUNDLES = {
    'main': REMOTE_SEED_ROOT + '/seed36_best_weights.zip',       # YOLO11m/26m
    'raw_n': REMOTE_SEED_ROOT + '/raw_yolo/seed36_best_weights.zip',
    'curated_n': REMOTE_SEED_ROOT + '/curated_yolo/seed36_best_weights.zip',
}
RUNS = ['raw_yolo11n','curated_yolo11n','raw_yolo26n','curated_yolo26n',
        'raw_yolo11m','curated_yolo11m','raw_yolo26m','curated_yolo26m']
SELECTED = list(RUNS)  # edit to test a subset
IMGSZ, BATCH, DEVICE, WORKERS = 640, 8, 0, 2
print('Test:', TEST_YAML)
print('Selected:', SELECTED)
print('Local results:', RESULTS_ROOT)
print('Camber results:', REMOTE_RESULTS_ROOT)

## Camber helpers and checkpoint preparation

In [ ]:
def load_camber_key():
    if os.environ.get('CAMBER_API_KEY'):
        return
    for line in (REPO/'.env').read_text(encoding='utf-8').splitlines():
        key, sep, value = line.partition('=')
        if sep and key.strip() == 'CAMBER_API_KEY' and value.strip():
            os.environ['CAMBER_API_KEY'] = value.strip().strip(chr(34)).strip(chr(39)); return
    raise RuntimeError('CAMBER_API_KEY missing from repository .env')

def wsl_path(path: Path) -> str:
    path = path.resolve()
    return f'/mnt/{path.drive.rstrip(chr(58)).lower()}{path.as_posix().split(chr(58),1)[1]}'

def camber_cp(source, destination, attempts=3):
    load_camber_key(); env = os.environ.copy()
    parts = [p for p in env.get('WSLENV','').split(':') if p]
    if 'CAMBER_API_KEY' not in parts: parts.append('CAMBER_API_KEY')
    env['WSLENV'] = ':'.join(parts)
    src = source if str(source).startswith('stash://') else wsl_path(Path(source))
    dst = destination if str(destination).startswith('stash://') else wsl_path(Path(destination))
    cmd = ['wsl','--','/home/dotru/.camber/bin/camber','stash','cp',str(src),str(dst)]
    for attempt in range(1, attempts+1):
        proc = subprocess.run(cmd, env=env, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
        if proc.returncode == 0: return
        if attempt < attempts: time.sleep(5*attempt)
    raise RuntimeError(proc.stdout[-3000:])

def upload_folder(local_dir: Path, remote_dir: str):
    for path in sorted(local_dir.rglob('*')):
        if path.is_file() and path.suffix != '.cache':
            camber_cp(path, f'{remote_dir}/{path.relative_to(local_dir).as_posix()}')

WEIGHTS_ROOT.mkdir(parents=True, exist_ok=True)
bundle_paths = {}
for key, remote in REMOTE_BUNDLES.items():
    local = WEIGHTS_ROOT/f'{key}_seed36_best_weights.zip'
    # Reuse the main bundle downloaded by the previous notebook revision.
    legacy = WEIGHTS_ROOT/'seed36_best_weights.zip'
    if key == 'main' and legacy.exists() and not local.exists(): local = legacy
    if not local.exists():
        print('Downloading', remote)
        camber_cp(remote, local)
    bundle_paths[key] = local

# Each run is looked up only in its authoritative archive. The main archive
# contains m checkpoints; the two child archives preserve the earlier n runs.
RUN_BUNDLE = {run: ('main' if run.endswith('m') else ('raw_n' if run.startswith('raw_') else 'curated_n')) for run in RUNS}
WEIGHTS = {}
for run in RUNS:
    bundle = bundle_paths[RUN_BUNDLE[run]]
    with zipfile.ZipFile(bundle) as archive:
        members = [m for m in archive.namelist() if not m.endswith('/')]
        matches = [m for m in members if run.lower() in m.replace('\\','/').lower() and Path(m).name.lower() == 'best.pt']
        if len(matches) != 1:
            candidates = [m for m in members if Path(m).name.lower() == 'best.pt']
            raise RuntimeError(f'{bundle.name}: expected one best.pt for {run}; found {matches}; available={candidates}')
        target = WEIGHTS_ROOT/run/'best.pt'; target.parent.mkdir(parents=True, exist_ok=True)
        if not target.exists():
            with archive.open(matches[0]) as src, target.open('wb') as dst: shutil.copyfileobj(src,dst)
        WEIGHTS[run] = target

for run, path in WEIGHTS.items(): print(f'{run:16} {path.stat().st_size/1024**2:7.1f} MiB')

## Read-only test and checkpoint preflight

In [ ]:
cfg = yaml.safe_load(TEST_YAML.read_text(encoding='utf-8'))
freeze = json.loads((TEST_ROOT/'freeze.json').read_text(encoding='utf-8'))
assert Path(cfg['path']).resolve() == TEST_ROOT.resolve(), (cfg['path'], TEST_ROOT)
assert cfg['test'] == 'images/test' and len(cfg['names']) == 6
images = [p for p in (TEST_ROOT/'images/test').iterdir() if p.is_file() and p.suffix != '.cache']
labels = list((TEST_ROOT/'labels/test').glob('*.txt'))
assert len(images) == len(labels) == 1931, (len(images),len(labels))
assert freeze['name'] == 'test_main_v5_full_consensus_assisted', freeze.get('name')
assert SELECTED and not (set(SELECTED)-set(RUNS)), SELECTED
for run in SELECTED:
    assert WEIGHTS[run].is_file() and WEIGHTS[run].stat().st_size > 4_000_000, (run,WEIGHTS[run])
print(f'Preflight OK: {len(images)} images, {len(labels)} labels, {len(SELECTED)} checkpoints')

## Run selected evaluations

In [ ]:
from ultralytics import YOLO

def sha256(path: Path):
    h=hashlib.sha256()
    with path.open('rb') as f:
        for chunk in iter(lambda:f.read(1024*1024),b''): h.update(chunk)
    return h.hexdigest()

def evaluate_one(run):
    weight=WEIGHTS[run].resolve(); out=RESULTS_ROOT/run; out.mkdir(parents=True,exist_ok=True)
    print(f'[{run}] audit V5',flush=True)
    metrics=YOLO(str(weight)).val(data=str(TEST_YAML),split='test',imgsz=IMGSZ,batch=BATCH,
        device=DEVICE,workers=WORKERS,project=str(RESULTS_ROOT),name=run,exist_ok=True,plots=True,save_json=False)
    names=metrics.names if isinstance(metrics.names,dict) else dict(enumerate(metrics.names))
    row={'run':run,'seed':36,'dataset':'raw' if run.startswith('raw_') else 'curated',
         'model':run.split('_',1)[1],'weight_sha256':sha256(weight),'test_images':1931,
         'precision':float(metrics.box.mp),'recall':float(metrics.box.mr),
         'map50':float(metrics.box.map50),'map50_95':float(metrics.box.map),
         'speed_ms':{k:float(v) for k,v in metrics.speed.items()},
         'per_class_map50_95':{names[i]:float(v) for i,v in enumerate(metrics.box.maps)}}
    (out/'metrics.json').write_text(json.dumps(row,indent=2),encoding='utf-8')
    upload_folder(out,f'{REMOTE_RESULTS_ROOT}/{run}')
    return row

records=[evaluate_one(run) for run in SELECTED]

## Build and upload combined summary

In [ ]:
available=[]
for run in RUNS:
    path=RESULTS_ROOT/run/'metrics.json'
    if path.is_file(): available.append(json.loads(path.read_text(encoding='utf-8')))
assert available, 'Run at least one evaluation first'
available.sort(key=lambda row:row['map50_95'],reverse=True)
RESULTS_ROOT.mkdir(parents=True,exist_ok=True)
summary_json=RESULTS_ROOT/'summary.json'; summary_csv=RESULTS_ROOT/'summary.csv'
summary_json.write_text(json.dumps(available,indent=2),encoding='utf-8')
fields=['run','seed','dataset','model','weight_sha256','test_images','precision','recall','map50','map50_95']
with summary_csv.open('w',newline='',encoding='utf-8') as f:
    w=csv.DictWriter(f,fieldnames=fields); w.writeheader(); w.writerows({k:r[k] for k in fields} for r in available)
camber_cp(summary_json,f'{REMOTE_RESULTS_ROOT}/summary.json')
camber_cp(summary_csv,f'{REMOTE_RESULTS_ROOT}/summary.csv')
for r in available: print(f"{r['run']:16} mAP50={r['map50']:.4f} mAP50-95={r['map50_95']:.4f}")